# ESdE Adultos 2023: Test Exploratory analysis

~~This notebook is for exploratory work with the extracted INE microdata. Reusable extraction, loading, and codebook functions stay in `ine_health_data/`; calculations and visual inspection belong here.~~

Run the notebook from the repository root. 

The raw ESdE file and `references/metadata/esde_adulto_2023.json` must already exist. Otherwise run `ine_health_data.pipeline.start_sequence()`

In [1]:
import pandas as pd
from pathlib import Path
from ine_health_data.pipeline import CODEBOOK_OUTPUT_PATH, add_value_labels, load_variables

## General population analysis
(population represented)

Stablishing the population that the dataset represents before realizing any in depth analysis.

Main target variables should be age, sex, and location.

In [2]:
analysis_variable = ["EDADa","SEXOa","CCAA"] 
raw_data = load_variables(variables=analysis_variable)

### Data integrity

The variable `EDADa` is the only numeric variable loaded, which includes special `999` code for "Not answered" value.
For `SEXOa`,`CCAA`, There are no possible "Not answered" special values.

In [38]:
rows_count          = len(raw_data)
nonfull_rows_count  = raw_data.isna().any(axis=1).sum()
age_not_answered_c  = raw_data["EDADa"].eq(999).sum()

pd.Series({
    "Total rows": rows_count,
    "Rows any with empty": nonfull_rows_count,
    "Not answered age rows": age_not_answered_c
}).to_frame(name="n")

,n
Total rows,21032
Rows any with empty,0
Not answered age rows,0


The dataset for `SEXOa`, `CCAA` and `EDADa` does **not** contain any empty or "Not answered" value.

This implication is carried into following python cells, avoiding the need for unnecessary filtering.

### General Age and Sex analysis

In [ ]:
non_ccaa_df = raw_data[["EDADa","SEXOa"]]

age_summary = (
    non_ccaa_df.loc[non_ccaa_df["EDADa"].ne(999), "EDADa"]
    .describe(percentiles=[0.25,0.5,0.75])
    .to_frame().T
    .rename(index={"EDADa": "General age data"})
    .round(2)
)
display(age_summary)

labeled_national_df = add_value_labels(non_ccaa_df)

age_by_sex_freq = labeled_national_df["SEXOa_label"].value_counts(normalize=True).round(4)
age_by_sex_desc = (
    labeled_national_df.groupby("SEXOa_label")["EDADa"]
    .describe(percentiles=[0.25,0.5,0.75])
    .round(2)
)
age_by_sex_summary = (
    pd.concat([age_by_sex_freq, age_by_sex_desc], axis=1)
    .reset_index()
    .rename(columns={"SEXOa_label": "sex"})
)
display(age_by_sex_summary)

,count,mean,std,min,25%,50%,75%,max
General age data,21032.0,54.49,19.14,15.0,40.0,55.0,69.0,103.0


,sex,proportion,count,mean,std,min,25%,50%,75%,max
0,Mujer,0.5397,11352.0,55.92,19.49,15.0,41.0,56.0,71.0,103.0
1,Hombre,0.4603,9680.0,52.8,18.59,15.0,39.0,53.0,67.0,98.0


### Age and Sex per Location

In [41]:
df_labeled = add_value_labels(raw_data)

test = (
   df_labeled.groupby("CCAA_label")["EDADa"]
    .agg(["count","mean","median","min","max"])
    .round(2)
    .reset_index()
    # .rename(columns={"SEXOa_label": "sex"}) 
)
display(test)

,CCAA_label,count,mean,median,min,max
0,Andalucía,2674,52.67,53.0,15,99
1,Aragón,1044,55.73,56.0,15,99
2,"Asturias, Principado de",705,58.16,59.0,15,99
3,"Balears, Illes",680,52.59,52.0,15,98
4,Canarias,874,52.3,53.0,15,98
5,Cantabria,635,55.45,56.0,15,97
6,Castilla - La Mancha,1030,55.31,55.0,15,95
7,Castilla y León,1344,57.05,58.0,15,99
8,Cataluña,1802,54.32,55.0,15,102
9,Ceuta,203,49.22,48.0,15,93
